# Build and benchmark a Wordle environment

This brief, offline walkthrough shows the full Enroute loop:

1. inspect a tool-based environment,
2. apply actions through framework-owned `step()`,
3. run and inspect a scored episode trace,
4. define a versioned `TaskDataset`, and
5. compare policy factories with `Benchmark`.

`WordleEnv` stays true to the runnable example: the environment owns the secret and board, while the policy only chooses `guess` tool calls.

In [ ]:
from pathlib import Path

from enroute import Benchmark, JSONLSink, ScriptedPolicy, TaskData, TaskDataset, TraceWriter
from enroute.tracing import ParsedAction
from examples.environment.wordle.env import MAX_GUESSES, make_env

env = make_env()
print(f"{env.name}@{env.version}  fingerprint={env.fingerprint()[:12]}…")
print("tools:", [tool.function.name for tool in env.tool_defs])
print("default tasks:", [task.task_id for task in env.iter_tasks()])

## 1. Apply one tool action

Tool-based environments define `@tool` methods and normally inherit `apply_action()`. `Environment.step()` is final and framework-owned: it validates lifecycle state, executes tools, records a `Decision`, adds rewards, rebuilds the observation, and determines termination.

For a custom scalar or text action space, override `apply_action()` and return `ActionResult`—never override `step()`.

In [ ]:
task = TaskData(
    task_id="notebook-crane",
    input="Play Wordle using the guess tool.",
    expected="crane",  # hidden from the policy; used to seed this example
    metadata={"seed": 0},
)

observation, reset_info = env.reset(task, model="manual-policy")
result = env.step(ParsedAction(name="guess", arguments={"word": "slate"}))
print(result.observation.board)
print("dense reward:", result.reward, "turn:", result.info["turn"])
print("guess slots left:", result.observation.guesses_left, "of", MAX_GUESSES)
manual_rollout = env.close_episode()

## 2. Run and inspect a complete episode

`run_episode()` is the easiest offline path for a custom policy. A trace stores the policy-visible observation, parsed action, tool result, dense rewards, final score, stop reason, and environment fingerprint. `returns()` converts those rewards into per-decision training targets.

In [ ]:
solver = ScriptedPolicy(
    [
        ParsedAction(name="guess", arguments={"word": "slate"}),
        ParsedAction(name="guess", arguments={"word": "crane"}),
    ]
)
rollout = make_env().run_episode(task, solver, model="scripted-solver")

for decision in rollout.trace.decisions():
    action = decision.parsed_action[0]
    reward = sum(event.value for event in decision.reward_events)
    print(decision.index, action.name, action.arguments, "dense_reward=", reward)
print("terminal reward:", rollout.trace.outcome.reward)
print("returns:", rollout.trace.returns(gamma=0.9, source="both"))

## 3. Benchmark policies on fixed tasks

A benchmark needs stable tasks and fresh policy/environment instances for every case. `TaskDataset` hashes task inputs—including hidden `expected` labels—while policy factories prevent mutable state from leaking across concurrent runs. A caller-owned `TraceWriter` makes policy episode traces durable.

In [ ]:
dataset = TaskDataset(name="wordle-notebook", version="1.0.0", tasks=[task])
misses = ["audio", "wordy", "aback", "abase", "abate", "abbey"]

policies = {
    "solver": lambda: ScriptedPolicy(
        [
            ParsedAction(name="guess", arguments={"word": "slate"}),
            ParsedAction(name="guess", arguments={"word": "crane"}),
        ]
    ),
    "baseline": lambda: ScriptedPolicy(
        [ParsedAction(name="guess", arguments={"word": word}) for word in misses]
    ),
}

out = Path(".enroute/examples")
out.mkdir(parents=True, exist_ok=True)
writer = TraceWriter(JSONLSink(out / "wordle-notebook-episodes.jsonl"))
try:
    report = Benchmark.from_policies(
        make_env(),
        policies,
        concurrency=2,
        environment_factory=make_env,
        trace_writer=writer,
    ).run(dataset=dataset)
finally:
    writer.close()

print(report.to_markdown())

## Adapt this pattern to your task

1. Define typed `State` for simulator internals and `Observation` for policy-visible data.
2. Implement `setup(task)`, `observe()`, `done()`, and a trace-safe `snapshot()`.
3. Add actions with `@tool`. For a non-tool action space, override `apply_action()` and return `ActionResult`; do not override `step()`.
4. Register terminal scorers and optional dense rewards through `step_reward()`.
5. Give every `TaskData` a stable `task_id`; pin randomness in metadata and keep evaluator-only labels in `expected`.
6. Test `run_episode()`, trace rewards, stop reasons, and replay before scaling up.
7. Benchmark custom agents with `Benchmark.from_policies()`. To benchmark real models, create an `Enroute` client and use `Benchmark(env, models=[...], client=client).run(dataset=dataset)`.

See `env.py` for the complete environment and `run.py` for local or hosted model execution.